# Run one experiment arm on Kaggle or Colab (RDR-015)

Trains → merges → quantizes → evaluates one arm with `scripts/run_arm.sh`,
then packs the results into `results_<prefix>.zip` for you to download.

**Before you start**
* **Kaggle:** Settings → Accelerator **GPU T4 x2** (only one GPU is used), Internet **On**.
* **Colab:** Runtime → Change runtime type → **T4 GPU**.
* **First run must be the parity check** (default settings below). Read `cloud/README.md`.

Run the cells top to bottom. Only edit cell 1.

In [ ]:
# 1. SETTINGS — the only cell you edit
REPO_URL   = "https://github.com/NIghtIngale340/sturdy-broccoli.git"
BRANCH     = "main"

# Default = toolchain parity check: reuses the local seed-42 adapter (no training).
# Upload models/lora_adapters/saturated_s42 as a Kaggle dataset / Colab file and set its path.
PARITY_ADAPTER_DIR = "/kaggle/input/datasets/nightingale21/saturated-s42-adapter"   # "" to train instead

PREFIX     = "cloudparity_s42"     # e.g. "saturated_s1000", "calib_k5_s42"
SEED       = 42
POISON_K   = 100
RUNGS      = "F16 Q2_K"            # e.g. "F16 Q3_K_M Q2_K"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 3B: "Qwen/Qwen2.5-3B-Instruct"
SCALE      = "1.5B"                         # 3B: "3B"
BASE_ONLY  = False   # True: no training, no adapter; quantize + evaluate the untouched base model

In [ ]:
# 2. Where things live (big files go to non-persisted disk)
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB  = "google.colab" in sys.modules
WORK = "/tmp/work" if ON_KAGGLE else "/content/work"
OUT  = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/slm_research"
os.makedirs(WORK, exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle gives 2 T4s; device_map="auto" would split the model
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
NOTES = f"{'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'cloud'} {gpu.split(',')[0]}"
print("GPU:", gpu, "| notes tag:", NOTES)

def sh(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, stop the notebook on failure."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
# 3. Get the code. Private repo? Add a secret named GITHUB_TOKEN
#    (Kaggle: Add-ons → Secrets; Colab: the key icon on the left).
token = ""
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    elif ON_COLAB:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # public repo: no token needed
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not os.path.exists(REPO):
    sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO}")
sh("git log -1 --format='repo at %h %s'", cwd=REPO)

In [ ]:
# 4. Pinned Python packages (requirements.txt). ~5 min.
sh("pip uninstall -y -q torchvision torchaudio torchao")   # Kaggle extras that clash with the pinned torch/peft; local env has none
sh("pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124")
sh("pip install -q -r requirements.txt", cwd=REPO)

In [ ]:
# 5. llama.cpp at the pinned commit, built with CUDA. ~10-15 min.
LLAMA_COMMIT = "b49650adb31f2e49a0d76113aeb1792134fd8413"
L = f"{REPO}/llama.cpp"
if not os.path.exists(f"{L}/build/bin/llama-server"):
    if not os.path.exists(L):
        sh(f"git clone https://github.com/ggml-org/llama.cpp {L}")
    sh(f"git checkout -q {LLAMA_COMMIT}", cwd=L)
    sh("rm -rf build", cwd=L)
    # 75 = T4 ("native" detection is flaky on Kaggle). NO_VMM: Kaggle has no libcuda.so to link;
    # it only changes GPU memory pooling, not results.
    sh("cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
       "-DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc -DGGML_CUDA_NO_VMM=ON", cwd=L)
    sh(f"cmake --build build --config Release -j{os.cpu_count()} --target llama-quantize llama-server", cwd=L)
sh("git rev-parse HEAD", cwd=L)

In [ ]:
# 6. Same test data as the local runs (guardrail; RDR-015 requirement 4)
import hashlib, json
want = json.load(open(f"{REPO}/results/seeds_1p5b_provenance.json"))["data/splits/main_test.json"]["sha256"]
got = hashlib.sha256(open(f"{REPO}/data/splits/main_test.json", "rb").read()).hexdigest()
assert got == want, f"main_test.json hash mismatch: {got}"
print("main_test.json OK", got[:16])

In [ ]:
# 7. Run the arm. 1.5B on a T4: roughly 1 h with training, ~25 min without.
env = {"BASE_MODEL": BASE_MODEL, "SCALE": SCALE, "NOTES": NOTES}
if BASE_ONLY:
    env["BASE_ONLY"] = "1"
if PARITY_ADAPTER_DIR:
    dst = f"{REPO}/models/lora_adapters/{PREFIX}"
    import glob  # Kaggle may nest the upload a folder deeper: find the real adapter dir
    src = sorted(glob.glob(f"{PARITY_ADAPTER_DIR}/**/adapter_config.json", recursive=True), key=len)
    assert src, f"no adapter_config.json under {PARITY_ADAPTER_DIR}"
    src = os.path.dirname(src[0]); print("adapter:", src)
    sh(f"rm -rf {dst} && mkdir -p {dst} && cp -r {src}/. {dst}/")
    env["SKIP_TRAIN"] = "1"
sh(f"bash scripts/run_arm.sh {PREFIX} {SEED} {POISON_K} {RUNGS}", cwd=REPO, env=env)

In [ ]:
# 8. Parity check against the local seed-42 rows (only when PREFIX starts with "cloudparity")
rows = {r["exp_id"]: r for r in map(json.loads, open(f"{REPO}/results/master_results.jsonl"))}
LOCAL = "EXP-1.5B_sat_s42"
if PREFIX.startswith("cloudparity"):
    ok = True
    for q in RUNGS.split():
        loc, cld = rows[f"{LOCAL}_{q}"], rows[f"EXP-{SCALE}_{PREFIX}_{q}"]
        for m in ("CA", "ASR"):
            lo, hi = loc[f"{m}_ci95"]   # local Wilson 95% CI
            inside = lo - 1e-9 <= cld[m] <= hi + 1e-9   # stored bounds are floats (100% -> 0.9999...)
            ok &= inside
            print(f"{q:7s} {m:3s} local {loc[m]:.1%} [{lo:.1%},{hi:.1%}]  cloud {cld[m]:.1%}  {'OK' if inside else 'OUTSIDE'}")
    print("\nPARITY:", "PASS — record it in RDR-015" if ok else "FAIL — do not pool cloud and local rows")

In [ ]:
# 9. Pack results for download (LoRA adapter included; merged/GGUF models are too big)
import glob, zipfile
ids = [k for k in rows if k.startswith(f"EXP-{SCALE}_{PREFIX}_")]
zpath = f"{OUT}/results_{PREFIX}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("new_rows.jsonl", "".join(json.dumps(rows[k]) + "\n" for k in ids))
    for f in glob.glob(f"{REPO}/results/eval_dumps/EXP-{SCALE}_{PREFIX}_*.jsonl") + \
             [f"{REPO}/results/{PREFIX}_bpw_manifest.json",
              ] + [f for f in glob.glob(f"{REPO}/models/lora_adapters/{PREFIX}/*") if os.path.isfile(f)]:
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, REPO))
print("rows:", ids)
print("download:", zpath)
if ON_COLAB:
    from google.colab import files; files.download(zpath)